In [ ]:
import json

import mlflow

from sarah_hotel_reservation_data.data_processing import HotelDataset

In [2]:
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(experiment_name="/Shared/hotel-data-sarah")
mlflow.set_experiment_tags({"repository_name": "hotel-data-sarah"})

experiments = mlflow.search_experiments(
    filter_string="tags.repository_name='hotel-data-sarah'"
)
print(experiments)

[<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/688553492365940', creation_time=1731077463760, experiment_id='688553492365940', last_update_time=1731077463760, lifecycle_stage='active', name='/Shared/hotel-data-sarah', tags={'mlflow.experiment.sourceName': '/Shared/hotel-data-sarah',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'sarah.glasmacher@swk.de',
 'mlflow.ownerId': '4197150212560385',
 'repository_name': 'hotel-data-sarah'}>]


In [3]:
with open("mlflow_experiment.json", "w") as json_file:
    json.dump(experiments[0].__dict__, json_file, indent=4)

## Load data with prepared functions

In [7]:
hotel_data = HotelDataset(
    data_filepath="./data/Hotel Reservations.csv",
    yaml_file_path="./hotel_data_config.yaml",
)
hotel_data.run_data_preparation()
X_train, y_train = hotel_data.get_train_data()
X_val, y_val = hotel_data.get_val_data()

X_train.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,required_car_parking_space,lead_time,arrival_year,arrival_month,arrival_date,repeated_guest,...,room_type_reserved_Room_Type 2,room_type_reserved_Room_Type 3,room_type_reserved_Room_Type 4,room_type_reserved_Room_Type 5,room_type_reserved_Room_Type 6,room_type_reserved_Room_Type 7,market_segment_type_Complementary,market_segment_type_Corporate,market_segment_type_Offline,market_segment_type_Online
6922,2.226761,-0.26147,-0.931190,1.272747,-0.178819,-0.282005,0.467843,-0.463753,-1.670074,-0.16221,...,False,False,True,False,False,False,False,False,False,True
32875,0.298893,-0.26147,-0.931190,-0.853578,-0.178819,-0.689315,0.467843,-0.463753,-0.869189,-0.16221,...,False,False,False,False,False,False,False,False,True,False
31424,0.298893,-0.26147,0.217401,1.981521,-0.178819,2.278230,0.467843,1.490739,1.190231,-0.16221,...,False,False,False,False,False,False,False,False,False,True
19532,2.226761,-0.26147,-0.931190,1.272747,-0.178819,-0.898788,0.467843,-1.766747,-0.754777,-0.16221,...,False,False,True,False,False,False,False,False,False,True
2088,0.298893,-0.26147,-0.931190,-1.562353,-0.178819,-0.852239,0.467843,0.187744,0.046109,-0.16221,...,False,False,False,False,False,False,False,False,False,True


## Model Experiments / offline

In [8]:
from sklearn.ensemble import RandomForestClassifier

In [9]:
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

/Users/sarahglasmacher/Documents/Coding/marvelous-databricks-course-GalaxyInfernoCodes/.venv/lib/python3.11/site-packages/sklearn/base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# 1. Generate predictions on validation set
y_pred = clf.predict(X_val)

# 2. Calculate evaluation metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(
    y_val, y_pred, average="binary"
)  # 'binary' for binary classification
recall = recall_score(y_val, y_pred, average="binary")
f1 = f1_score(y_val, y_pred, average="binary")

# 3. Print evaluation results
print("Model Evaluation Metrics on Validation Set:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# 4. Display the confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

# 5. Detailed classification report
print("\nClassification Report:")
print(classification_report(y_val, y_pred))

Model Evaluation Metrics on Validation Set:
Accuracy: 0.8978
Precision: 0.8771
Recall: 0.8047
F1 Score: 0.8393

Confusion Matrix:
[[3662  217]
 [ 376 1549]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.94      0.93      3879
           1       0.88      0.80      0.84      1925

    accuracy                           0.90      5804
   macro avg       0.89      0.87      0.88      5804
weighted avg       0.90      0.90      0.90      5804

